In [ ]:
import MDAnalysis as mda
import numpy as np


def calculate_electric_field(pdb_file, prmtop_file, atom1, atom2):
    # Load the universe
    u = mda.Universe(prmtop_file, pdb_file)

    # Constants
    e = 1.60218e-19  # Elementary charge in Coulombs
    epsilon_0 = 8.85419e-12  # Vacuum permittivity in F/m (C^2/(N*m^2))
    angstrom_to_meter = 1e-10

    # Select the atoms involved in the bond
    atom1 = u.select_atoms(f'name {atom1}')[0]
    atom2 = u.select_atoms(f'name {atom2}')[0]

    # Coordinates of the bond atoms
    pos1 = atom1.position * angstrom_to_meter
    pos2 = atom2.position * angstrom_to_meter

    # Vector along the bond
    bond_vector = pos2 - pos1
    bond_length = np.linalg.norm(bond_vector)
    bond_unit_vector = bond_vector / bond_length

    # Calculate the midpoint of the bond
    midpoint = (pos1 + pos2) / 2

    # Initialize electric field vector
    electric_field = np.zeros(3)

    # Calculate the electric field contribution from each atom in the environment
    for atom in u.atoms:
        if atom.index not in (atom1.index, atom2.index):
            charge = atom.charge * e  # Convert charge to Coulombs
            r = atom.position * angstrom_to_meter  # Position of the atom in meters
            r_vector = r - midpoint  # Vector from midpoint to atom
            # Distance from midpoint to atom
            r_distance = np.linalg.norm(r_vector)

            if r_distance != 0:
                # Electric field contribution from this atom
                E = (1 / (4 * np.pi * epsilon_0)) * \
                    (charge / r_distance**3) * r_vector
                electric_field += E

    # Project the electric field onto the bond direction
    electric_field_along_bond = np.dot(electric_field, bond_unit_vector)

    # Return the electric field along the bond
    return electric_field_along_bond


# Example usage
pdb_file = 'rc.opt.pdb'
prmtop_file = 'rc.prmtop'
atom1_name = 'CA'  # Replace with the first atom name in the bond
atom2_name = 'CB'  # Replace with the second atom name in the bond

electric_field_along_bond = calculate_electric_field(
    pdb_file, prmtop_file, atom1_name, atom2_name)
print(f'Electric field along the bond: {electric_field_along_bond} N/C')